# Run the HIF / CICIDS2017 comparison on Google Colab

This notebook clones the repository, installs the pinned dependencies,
downloads the dataset and runs the full comparison. It is meant to run on
Colab so you do not have to run the heavy pipeline on your own machine.

## Runtime: use CPU, not GPU

The entire pipeline (HIF, Random Forest, MLP, SVM, LOF) runs on CPU through
scikit-learn. There is NO GPU acceleration, so selecting a GPU runtime (T4,
L4, A100) gives zero speedup and only burns compute units. Pick:

    Runtime -> Change runtime type -> CPU, and enable High-RAM.

High-RAM matters because the dataset has millions of rows. More vCPUs help:
Random Forest, the HIF forest, LOF and the SVM calibration folds use all
cores. The MLP is the main step that does not parallelize within a fit; to
use spare cores for it during tuning, add `--parallel_trials 4` to the final
run (this makes the MLP's Optuna search non-deterministic, so it is off by
default).

## 1. Clone the repository
Edit REPO_URL if you forked it or pushed it elsewhere.

In [ ]:
REPO_URL = "https://github.com/Dicotomico23/hif-cicids2017.git"
import os
repo = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')
if not os.path.isdir(repo):
    !git clone $REPO_URL
%cd $repo
!git log --oneline -1

## 2. Install dependencies

In [ ]:
# Colab already ships numpy/pandas/scipy/scikit-learn/matplotlib. Install only
# the missing pieces and the package itself, without forcing version changes.
!pip install -q -U kagglehub imbalanced-learn optuna
!pip install -q -e . --no-deps
# Note: this uses Colab's library versions. For the exact pinned environment
# used in the paper, install requirements.txt locally instead.


## 3. Get the dataset

The next cell runs `data/download.py`, which downloads the cleaned dataset
from this GitHub Release asset and verifies its SHA256 checksum:

https://github.com/Dicotomico23/hif-cicids2017/releases/download/dataset-v1/cicids2017_cleaned.zip

No Kaggle account is needed. Kaggle stays as an optional emergency fallback
(commented at the bottom of the cell).

In [ ]:
!python data/download.py
import os
assert os.path.isfile('data/cicids2017_cleaned.csv'), (
    'Cleaned dataset is missing. Re-run this cell before the comparison; '
    'otherwise the pipeline falls back to the RAW Kaggle data (78 features '
    'instead of 52) and the numbers will not match the paper.')
print('Cleaned dataset ready (%.0f MB)'
      % (os.path.getsize('data/cicids2017_cleaned.csv') / 1e6))

## 4. Run the comparison

Two options. Results table and figures are written to `results/`.

- Quick pass (next cell): a small stratified fraction, no hyperparameter
  search. Completes in a few minutes and produces the full table and all
  figures. Use it to confirm everything runs end to end.
- Final run (cell after): a larger fraction with Optuna tuning. This is the
  heavy one (can take one to a few hours on Colab); raise `--fraction`
  toward `1.0` for the whole dataset, or lower it to trade scale for speed.

The pipeline is identical in both; only the data fraction and whether the
baselines are Optuna-tuned change.

In [ ]:
# Quick pass: small stratified fraction, no Optuna. Finishes in minutes.
!python reproduce/run_comparison.py --output results --fraction 0.05

In [ ]:
# Final run (heavy): larger fraction + Optuna tuning. Can take 1-3 h on Colab.
# Adjust --fraction (up to 1.0 = full dataset) and --n_trials to taste.
!python reproduce/run_comparison.py --output results --fraction 0.25 --optimize --n_trials 10

## 5. Show the results

In [ ]:
import pandas as pd
from IPython.display import Image, display
print(pd.read_csv('results/table5_results.csv').to_string(index=False))
for fig in ['fig_radar_metrics', 'fig_hif_confusion_matrix', 'fig_bar_balanced_acc', 'fig_bar_precision']:
    display(Image('results/figures/%s.png' % fig))